# Data cleaning: TetrapodTraits v3.0.1
Turns the raw TetrapodTraits table (33,281 species × 140 columns) into two files:

- `data/tetrapods_clean.csv`: every assessed, non-marine species, 29 columns. Used for the overview chart (Vis 1) and the map (Vis 4).
- `data/tetrapods_sample.csv`: a stratified sample of 10,000 species with a usable risk category and threat data.

In [1]:
import pandas as pd

RAW_PATH = 'data/TetrapodTraits_v3.0.1.csv'
CLEAN_PATH = 'data/tetrapods_clean.csv'
SAMPLE_PATH = 'data/tetrapods_sample.csv'
SAMPLE_SIZE = 10_000
SEED = 78

## Step 0: Load the raw data

In [2]:
# Read the original file unchanged
raw = pd.read_csv(RAW_PATH, low_memory=False)
raw.shape

(33281, 140)

## Step 1: Fix the IUCN risk category

In [3]:
df = raw.copy()

# Convert outdated pre-2001 'Lower Risk' codes to the current categories
legacy = {'LRlc': 'LC', 'LRnt': 'NT', 'LRcd': 'NT'}
df['Status'] = df['AssessedStatus'].replace(legacy)

# Drop species the IUCN has never assessed (no category at all)
df = df.dropna(subset=['Status'])

# Ordered risk score: 0 = Least Concern ... 4 = Critically Endangered
# (Data Deficient and Extinct get no score because they are not on this scale)
risk_order = {'LC': 0, 'NT': 1, 'VU': 2, 'EN': 3, 'CR': 4}
df['RiskScore'] = df['Status'].map(risk_order)

# Yes/no threatened label using the IUCN definition (VU, EN, or CR)
df['Threatened'] = df['Status'].isin(['VU', 'EN', 'CR'])

df['Status'].value_counts()

Status
LC    20888
DD     2936
EN     2675
VU     2491
NT     2102
CR     1485
EX      186
EW       10
Name: count, dtype: int64

## Step 2: Remove marine-only species

In [4]:
# Keep land and freshwater species
marine_only = (df['EcoMar'] == 1) & (df['EcoTer'] == 0) & (df['EcoFresh'] == 0)
print('Marine-only species removed:', marine_only.sum())
df = df[~marine_only]

Marine-only species removed: 134


## Step 3: Give coded columns readable values

In [5]:
# Plain-English group names for chart labels
df['Group'] = df['Class'].map({'Amphibia': 'Amphibians', 'Aves': 'Birds',
                               'Mammalia': 'Mammals', 'Reptilia': 'Reptiles'})

# Island endemic flag: 1/0 -> Island/Mainland
df['Island'] = df['Insularity'].map({1: 'Island', 0: 'Mainland'})

# Population trend from IUCN: 0/1/2 -> words (blank = no trend reported)
df['PopulationTrend'] = df['PopulationTrend'].map({0: 'Decreasing', 1: 'Stable', 2: 'Increasing'})

# Measured vs. estimated body mass: 1/0 -> True/False
df['ImputedMass'] = df['ImputedMass'].astype(bool)

## Step 4: Rename the 12 human-threat columns
Threat_1 to Threat_12 follow level 1 of the IUCN Threats Classification Scheme v3.3.

In [6]:
threat_names = {
    'Threat_1': 'Threat_Development',          # residential & commercial development
    'Threat_2': 'Threat_Agriculture',          # agriculture & aquaculture
    'Threat_3': 'Threat_EnergyMining',         # energy production & mining
    'Threat_4': 'Threat_Transport',            # roads, railways, shipping lanes
    'Threat_5': 'Threat_HuntingHarvest',       # biological resource use (hunting, logging, fishing)
    'Threat_6': 'Threat_HumanDisturbance',     # recreation, war, other human intrusion
    'Threat_7': 'Threat_NaturalSystemChange',  # fire suppression, dams
    'Threat_8': 'Threat_InvasiveDisease',      # invasive species, genes & diseases
    'Threat_9': 'Threat_Pollution',
    'Threat_10': 'Threat_Geological',          # volcanoes, earthquakes
    'Threat_11': 'Threat_ClimateChange',       # climate change & severe weather
    'Threat_12': 'Threat_Other',
}
df = df.rename(columns=threat_names | {'ThreatSum': 'ThreatCount'})
threat_cols = list(threat_names.values())

# Blank threat columns mean 'not recorded', not 'no threats'
# (every species with threat data has at least one), so they stay blank instead of being filled with 0
df['ThreatCount'].describe()

count    16751.000000
mean         2.705689
std          1.590777
min          1.000000
25%          1.000000
50%          2.000000
75%          4.000000
max         11.000000
Name: ThreatCount, dtype: float64

## Step 5: Rename the remaining columns we keep

In [7]:
df = df.rename(columns={
    'Scientific.Name': 'Species',
    'MajorHabitatSum': 'HabitatBreadth',   # number of IUCN habitat types used (already summarises 17 habitat columns)
})

## Step 6: Keep only the columns the project uses
Drops the 16 Source columns, body length (we use body mass), duplicate status columns, and the detailed habitat, diet, biome, and realm flags.

In [8]:
keep = [
    # identity and taxonomy
    'Species', 'Group', 'Order',
    # extinction risk
    'Status', 'RiskScore', 'Threatened', 'PopulationTrend',
    # ecological traits
    'BodyMass_g', 'ImputedMass', 'RangeSize_km2', 'HabitatBreadth', 'Island',
    # location (center of range) for the map
    'Longitude', 'Latitude',
    # human pressure inside the range
    'HumanDensity', 'PropCroplandArea',
    # human threats
    'ThreatCount', *threat_cols,
]
clean = df[keep].reset_index(drop=True)
clean.shape

(32639, 29)

## Step 7: Check the cleaned table

In [9]:
# No duplicate species
assert clean['Species'].is_unique

# Coordinates are valid where present (a handful of species have no range center; the map skips them)
assert clean['Latitude'].dropna().between(-90, 90).all() and clean['Longitude'].dropna().between(-180, 180).all()

# The threat count matches the 12 threat flags wherever threat data exists
has_threats = clean['ThreatCount'].notna()
assert (clean.loc[has_threats, threat_cols].sum(axis=1) == clean.loc[has_threats, 'ThreatCount']).all()

# Share of missing values per column (only threats, trend, and cropland should have gaps)
clean.isna().mean().round(3).loc[lambda s: s > 0]

RiskScore                     0.095
PopulationTrend               0.304
PropCroplandArea              0.012
ThreatCount                   0.487
Threat_Development            0.487
Threat_Agriculture            0.487
Threat_EnergyMining           0.487
Threat_Transport              0.487
Threat_HuntingHarvest         0.487
Threat_HumanDisturbance       0.487
Threat_NaturalSystemChange    0.487
Threat_InvasiveDisease        0.487
Threat_Pollution              0.487
Threat_Geological             0.487
Threat_ClimateChange          0.487
Threat_Other                  0.487
dtype: float64

## Step 8: Save the cleaned table

In [10]:
clean.to_csv(CLEAN_PATH, index=False)
print(f'Saved {CLEAN_PATH}: {clean.shape[0]:,} rows x {clean.shape[1]} columns')

Saved data/tetrapods_clean.csv: 32,639 rows x 29 columns


## Step 9: Build the stratified sample for the trait and threat charts
Only species with a usable category (not Data Deficient, not extinct) and threat data can answer the threat questions. From those, take 10,000 species while keeping each group's share and each group's threatened share the same.

Limitation to report: Least Concern species are much less likely to have threat data, so this subset leans toward threatened species. Use the full cleaned table, not this sample, for the share-threatened comparison (Vis 1).

In [11]:
# Species that can answer the threat questions
eligible = clean[clean['RiskScore'].notna() & clean['ThreatCount'].notna()]
print('Eligible species:', len(eligible))

# Stratified sample: same fraction from every Group x Threatened combination
frac = SAMPLE_SIZE / len(eligible)
sample = (eligible.groupby(['Group', 'Threatened'], group_keys=False)
                  .sample(frac=frac, random_state=SEED)
                  .reset_index(drop=True))

# Compare group mix and threatened share before vs. after sampling
pd.DataFrame({
    'eligible_share': eligible['Group'].value_counts(normalize=True),
    'sample_share': sample['Group'].value_counts(normalize=True),
    'eligible_threatened': eligible.groupby('Group')['Threatened'].mean(),
    'sample_threatened': sample.groupby('Group')['Threatened'].mean(),
}).round(3)

Eligible species: 15477


,eligible_share,sample_share,eligible_threatened,sample_threatened
Group,,,,
Amphibians,0.340,0.340,0.463,0.463
Birds,0.202,0.202,0.374,0.374
Mammals,0.176,0.176,0.447,0.447
Reptiles,0.281,0.281,0.382,0.382


## Step 10: Save the sample

In [12]:
sample.to_csv(SAMPLE_PATH, index=False)
print(f'Saved {SAMPLE_PATH}: {sample.shape[0]:,} rows x {sample.shape[1]} columns')

Saved data/tetrapods_sample.csv: 10,001 rows x 29 columns
